***************************
**MASTER POKEMON DATASET**
    <br/>https://github.com/lgreski/pokemonData/blob/master/Pokemon.csv</br>

Repository containing generations 1 - 9 of basic Pokémon stats, courtesy of pokemondb.net.

***************************
LICENSE: CC0 1.0 Universal

***************************
**NOTES:**
<br/>-	The purpose of this project is to determine stat distribution amongst the different generations, see if pokemon could be classified into different archetypes (e.g. glass cannon, tanks, etc.) to create the strongest, diverse team of 6. 
<br/>-	Multiple pokemon have different regional or form variations that share the same pokedex number. In order to avoid confusion when referencing the data, unique pokemon IDs were provided.
<br/>-	Resources:
		<br/>https://bulbapedia.bulbagarden.net/wiki/Main_Page
<br/>-   Archetypes:
		<br/>https://tvtropes.org/pmwiki/pmwiki.php/Main/FightingGameTropes
		<br/>https://www.reddit.com/r/rpg/comments/1byk5u3/are_there_any_archetypelike_terms_like_glass/
		<br/>https://pokemonpets.fandom.com/wiki/Walls/tanks/sweepers

In [1]:
import pandas as pd
import numpy as np
import statistics as stats
import random
import string

In [2]:
pokedex_file = r"D:\Data Analysis\Capstone\pokedex_clean_new.csv"
df = pd.read_csv(pokedex_file, encoding="latin-1")

In [3]:
# creating unique identifier
df['pokemon_id'] = ["PKM" + str(i).zfill(4) for i in range(1, len(df) + 1)]

df.sample(10)

,pokemon_id,pokedex_number,poke_name,form,type_1,type_2,total,hp,attack,defense,sp_attack,sp_defense,speed,generation
666,PKM0667,645,Landorus,Therian Forme,Ground,Flying,600,89,145,90,105,80,91,5
686,PKM0687,130,Gyarados,Mega Gyarados,Water,Dark,640,95,155,109,70,130,81,6
564,PKM0565,548,Petilil,,Grass,,280,45,35,50,70,50,30,5
80,PKM0081,81,Magnemite,,Electric,Steel,325,25,35,70,95,55,45,1
507,PKM0508,492,Shaymin,Land Forme,Grass,,600,100,100,100,100,100,100,4
490,PKM0491,479,Rotom,Wash Rotom,Electric,Water,520,50,65,107,105,107,86,4
1048,PKM1049,887,Dragapult,,Dragon,Ghost,600,88,120,75,100,75,142,8
428,PKM0429,419,Floatzel,,Water,,495,85,105,55,85,50,115,4
480,PKM0481,471,Glaceon,,Ice,,525,65,60,110,130,95,65,4
327,PKM0328,328,Trapinch,,Ground,,290,45,100,45,45,45,10,3


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1215 entries, 0 to 1214
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   pokemon_id      1215 non-null   object
 1   pokedex_number  1215 non-null   int64 
 2   poke_name       1215 non-null   object
 3   form            1215 non-null   object
 4   type_1          1215 non-null   object
 5   type_2          1215 non-null   object
 6   total           1215 non-null   int64 
 7   hp              1215 non-null   int64 
 8   attack          1215 non-null   int64 
 9   defense         1215 non-null   int64 
 10  sp_attack       1215 non-null   int64 
 11  sp_defense      1215 non-null   int64 
 12  speed           1215 non-null   int64 
 13  generation      1215 non-null   int64 
dtypes: int64(9), object(5)
memory usage: 133.0+ KB


In [ ]:
# exporting csv
# df.to_csv("pokedex_clean_new.csv", index=False, encoding="utf-8-sig")

### Analysis of Stats

In [5]:
# Average Stats

avg_hp = df['hp'].mean()
avg_atk = df['attack'].mean()
avg_def = df['defense'].mean()
avg_sp_atk = df['sp_attack'].mean()
avg_sp_def = df['sp_defense'].mean()
avg_speed = df['speed'].mean()

print(f'Average HP: {avg_hp}')
print(f'Average Attack: {avg_atk}')
print(f'Average Defense: {avg_def}')
print(f'Average Special Attack: {avg_sp_atk}')
print(f'Average Special Defense: {avg_sp_def}')
print(f'Average Speed: {avg_speed}')

Average HP: 71.24444444444444
Average Attack: 81.1522633744856
Average Defense: 75.00740740740741
Average Special Attack: 73.22469135802469
Average Special Defense: 72.44115226337449
Average Speed: 70.03456790123457


In [6]:
# Combining all stats into 1 df

avg_stats = df[['hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed']].mean()

print(avg_stats)    

hp            71.244444
attack        81.152263
defense       75.007407
sp_attack     73.224691
sp_defense    72.441152
speed         70.034568
dtype: float64


In [7]:
# Range of stats to observe outliers

hp_range = f'{df['hp'].min()}-{df['hp'].max()}'
atk_range = f'{df['attack'].min()}-{df['attack'].max()}'
def_range = f'{df['defense'].min()}-{df['defense'].max()}'
sp_atk_range = f'{df['sp_attack'].min()}-{df['sp_attack'].max()}'
sp_def_range = f'{df['sp_defense'].min()}-{df['sp_defense'].max()}'
speed_range = f'{df['speed'].min()}-{df['speed'].max()}'

print(f'HP range: {hp_range}')
print(f'Attack range: {atk_range}')
print(f'Defense range: {def_range}')
print(f'Special Attack range: {sp_atk_range}')
print(f'Special Defense range: {sp_def_range}')
print(f'Speed range: {speed_range}')

HP range: 1-255
Attack range: 5-190
Defense range: 5-250
Special Attack range: 10-194
Special Defense range: 20-250
Speed range: 5-200


### BCG Matrix

In [15]:
# BCG Matrix

# attack x speed
poke_atk_sp = df[
(df['attack'] > avg_stats['attack']) &
(df['speed'] > avg_stats['speed'])
]
print(poke_atk_sp[['pokemon_id','poke_name', 'attack', 'speed']])

# count per type 
atk_sp_count = poke_atk_sp.groupby('type_1').size().reset_index(name='count')
print(atk_sp_count)

     pokemon_id     poke_name  attack  speed
2       PKM0003      Venusaur      82     80
5       PKM0006     Charizard      84    100
8       PKM0009     Blastoise      83     78
14      PKM0015      Beedrill      90     75
21      PKM0022        Fearow      90    100
...         ...           ...     ...    ...
1207    PKM1208  Gouging Fire     115     91
1209    PKM1210  Iron Boulder     120    124
1212    PKM1213     Terapagos      95     85
1213    PKM1214     Terapagos     105     85
1214    PKM1215     Pecharunt      88     88

[338 rows x 4 columns]
      type_1  count
0        Bug     18
1       Dark     21
2     Dragon     28
3   Electric     22
4      Fairy      5
5   Fighting     25
6       Fire     26
7     Flying      5
8      Ghost      6
9      Grass     25
10    Ground     14
11       Ice      7
12    Normal     33
13    Poison     14
14   Psychic     27
15      Rock     16
16     Steel      9
17     Water     37


In [16]:
# attack x defense 
poke_atk_def = df[
(df['attack'] > avg_stats['attack']) &
(df['defense'] > avg_stats['defense'])
]
print(poke_atk_def[['pokemon_id','poke_name', 'attack', 'defense']])


# count per type 
atk_def_count = poke_atk_def.groupby('type_1').size().reset_index(name='count')
print(atk_def_count)

# cross check with excel
# bug_atk_def = poke_atk_def[poke_atk_def['type_1'] == 'Bug']
# print(bug_atk_def[['pokemon_id', 'poke_name', 'attack', 'defense']])

# water_atk_def = poke_atk_def[poke_atk_def['type_1'] == 'Water']
# print(water_atk_def[['pokemon_id', 'poke_name', 'attack', 'defense']])

     pokemon_id     poke_name  attack  defense
2       PKM0003      Venusaur      82       83
5       PKM0006     Charizard      84       78
8       PKM0009     Blastoise      83      100
27      PKM0028     Sandslash     100      110
30      PKM0031     Nidoqueen      92       87
...         ...           ...     ...      ...
1207    PKM1208  Gouging Fire     115      121
1209    PKM1210  Iron Boulder     120       80
1212    PKM1213     Terapagos      95      110
1213    PKM1214     Terapagos     105      110
1214    PKM1215     Pecharunt      88      160

[332 rows x 4 columns]
      type_1  count
0        Bug     20
1       Dark     18
2     Dragon     23
3   Electric      9
4      Fairy      5
5   Fighting     26
6       Fire     18
7     Flying      3
8      Ghost     12
9      Grass     28
10    Ground     20
11       Ice     10
12    Normal     20
13    Poison     13
14   Psychic     19
15      Rock     26
16     Steel     25
17     Water     37


In [17]:
# defense x hp
poke_def_hp = df[
(df['defense'] >= avg_stats['defense']) &
(df['hp'] >= avg_stats['hp'])
]
print(poke_def_hp[['pokemon_id','poke_name', 'defense', 'hp']])

# count per type 
def_hp_count = poke_def_hp.groupby('type_1').size().reset_index(name='count')
print(def_hp_count)

     pokemon_id   poke_name  defense   hp
2       PKM0003    Venusaur       83   80
5       PKM0006   Charizard       78   78
8       PKM0009   Blastoise      100   79
27      PKM0028   Sandslash      110   75
30      PKM0031   Nidoqueen       87   90
...         ...         ...      ...  ...
1210    PKM1211  Iron Crown      100   90
1211    PKM1212   Terapagos       85   90
1212    PKM1213   Terapagos      110   95
1213    PKM1214   Terapagos      110  160
1214    PKM1215   Pecharunt      160   88

[327 rows x 4 columns]
      type_1  count
0        Bug      9
1       Dark     16
2     Dragon     24
3   Electric     11
4      Fairy      9
5   Fighting     20
6       Fire     20
7     Flying      3
8      Ghost      7
9      Grass     29
10    Ground     22
11       Ice     13
12    Normal     31
13    Poison     12
14   Psychic     24
15      Rock     20
16     Steel     17
17     Water     40


In [18]:
# sp_attack x sp_defense
poke_sp_atk_def = df[
(df['sp_defense'] >= avg_stats['sp_defense']) &
(df['sp_attack'] >= avg_stats['sp_attack'])
]
print(poke_sp_atk_def[['pokemon_id','poke_name', 'sp_defense', 'sp_attack']])

# count per type 
sp_atk_def_count = poke_sp_atk_def.groupby('type_1').size().reset_index(name='count')
print(sp_atk_def_count)

     pokemon_id    poke_name  sp_defense  sp_attack
1       PKM0002      Ivysaur          80         80
2       PKM0003     Venusaur         100        100
5       PKM0006    Charizard          85        109
8       PKM0009    Blastoise         105         85
11      PKM0012   Butterfree          80         90
...         ...          ...         ...        ...
1208    PKM1209  Raging Bolt          89        137
1210    PKM1211   Iron Crown         108        122
1212    PKM1213    Terapagos         110        105
1213    PKM1214    Terapagos         110        130
1214    PKM1215    Pecharunt          88         88

[367 rows x 4 columns]
      type_1  count
0        Bug     11
1       Dark     12
2     Dragon     27
3   Electric     34
4      Fairy     15
5   Fighting      6
6       Fire     30
7     Flying      4
8      Ghost     16
9      Grass     40
10    Ground      7
11       Ice     14
12    Normal     25
13    Poison      9
14   Psychic     45
15      Rock     11
16     Steel

### Finding Pokemon Above Average Stats

In [9]:
# above hp

hp_poke = df[(df['hp'] > avg_stats['hp'])]
print(hp_poke[['pokemon_id','poke_name', 'hp']])
print(avg_stats['hp'])

     pokemon_id   poke_name   hp
2       PKM0003    Venusaur   80
5       PKM0006   Charizard   78
8       PKM0009   Blastoise   79
17      PKM0018     Pidgeot   83
27      PKM0028   Sandslash   75
...         ...         ...  ...
1210    PKM1211  Iron Crown   90
1211    PKM1212   Terapagos   90
1212    PKM1213   Terapagos   95
1213    PKM1214   Terapagos  160
1214    PKM1215   Pecharunt   88

[525 rows x 3 columns]
71.24444444444444


In [10]:
# above attack

atk_poke = df[(df['attack'] > avg_stats['attack'])]
print(atk_poke[['pokemon_id','poke_name', 'attack']])
print(avg_stats['attack'])

     pokemon_id     poke_name  attack
2       PKM0003      Venusaur      82
5       PKM0006     Charizard      84
8       PKM0009     Blastoise      83
14      PKM0015      Beedrill      90
21      PKM0022        Fearow      90
...         ...           ...     ...
1207    PKM1208  Gouging Fire     115
1209    PKM1210  Iron Boulder     120
1212    PKM1213     Terapagos      95
1213    PKM1214     Terapagos     105
1214    PKM1215     Pecharunt      88

[554 rows x 3 columns]
81.1522633744856


In [11]:
# above defense

def_poke = df[(df['defense'] > avg_stats['defense'])]
print(def_poke[['pokemon_id','poke_name', 'defense']])
print(avg_stats['defense'])

     pokemon_id   poke_name  defense
2       PKM0003    Venusaur       83
5       PKM0006   Charizard       78
7       PKM0008   Wartortle       80
8       PKM0009   Blastoise      100
26      PKM0027   Sandshrew       85
...         ...         ...      ...
1210    PKM1211  Iron Crown      100
1211    PKM1212   Terapagos       85
1212    PKM1213   Terapagos      110
1213    PKM1214   Terapagos      110
1214    PKM1215   Pecharunt      160

[516 rows x 3 columns]
75.00740740740741


In [12]:
# above sp_attack

sp_atk_poke = df[(df['sp_attack'] > avg_stats['sp_attack'])]
print(sp_atk_poke[['pokemon_id','poke_name', 'sp_attack']])
print(avg_stats['sp_attack'])

     pokemon_id    poke_name  sp_attack
1       PKM0002      Ivysaur         80
2       PKM0003     Venusaur        100
4       PKM0005   Charmeleon         80
5       PKM0006    Charizard        109
8       PKM0009    Blastoise         85
...         ...          ...        ...
1208    PKM1209  Raging Bolt        137
1210    PKM1211   Iron Crown        122
1212    PKM1213    Terapagos        105
1213    PKM1214    Terapagos        130
1214    PKM1215    Pecharunt         88

[526 rows x 3 columns]
73.22469135802469


In [13]:
# above sp_defense

sp_def_poke = df[(df['sp_defense'] > avg_stats['sp_defense'])]
print(sp_def_poke[['pokemon_id','poke_name', 'sp_defense']])
print(avg_stats['sp_defense'])

     pokemon_id   poke_name  sp_defense
1       PKM0002     Ivysaur          80
2       PKM0003    Venusaur         100
5       PKM0006   Charizard          85
7       PKM0008   Wartortle          80
8       PKM0009   Blastoise         105
...         ...         ...         ...
1210    PKM1211  Iron Crown         108
1211    PKM1212   Terapagos          85
1212    PKM1213   Terapagos         110
1213    PKM1214   Terapagos         110
1214    PKM1215   Pecharunt          88

[556 rows x 3 columns]
72.44115226337449


In [22]:
# above speed

speed_poke = df[(df['speed'] > avg_stats['speed'])]
print(speed_poke[['pokemon_id','poke_name', 'speed']])
print(avg_stats['speed'])

     pokemon_id     poke_name  speed
2       PKM0003      Venusaur     80
4       PKM0005    Charmeleon     80
5       PKM0006     Charizard    100
8       PKM0009     Blastoise     78
14      PKM0015      Beedrill     75
...         ...           ...    ...
1209    PKM1210  Iron Boulder    124
1210    PKM1211    Iron Crown     98
1212    PKM1213     Terapagos     85
1213    PKM1214     Terapagos     85
1214    PKM1215     Pecharunt     88

[550 rows x 3 columns]
70.03456790123457


In [23]:
# above all stats

all_poke = df[
(df['hp'] > avg_stats['hp']) &
(df['attack'] > avg_stats['attack']) &
(df['defense'] > avg_stats['defense']) &
(df['sp_attack'] > avg_stats['sp_attack']) &
(df['sp_defense'] > avg_stats['sp_defense']) &
(df['speed'] > avg_stats['speed'])
]

print(all_poke[['pokemon_id','poke_name', 'form', 'generation', 'hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed']])

     pokemon_id     poke_name           form  generation   hp  attack  \
2       PKM0003      Venusaur                          1   80      82   
5       PKM0006     Charizard                          1   78      84   
8       PKM0009     Blastoise                          1   79      83   
30      PKM0031     Nidoqueen                          1   90      92   
33      PKM0034      Nidoking                          1   81     102   
...         ...           ...            ...         ...  ...     ...   
1192    PKM1193      Miraidon                          9  100      85   
1193    PKM1194  Walking Wake                          9   99      83   
1212    PKM1213     Terapagos  Terastal Form           9   95      95   
1213    PKM1214     Terapagos   Stellar Form           9  160     105   
1214    PKM1215     Pecharunt                          9   88      88   

      defense  sp_attack  sp_defense  speed  
2          83        100         100     80  
5          78        109       

### Above Average Randomizer

In [20]:
avg_stats

hp            71.244444
attack        81.152263
defense       75.007407
sp_attack     73.224691
sp_defense    72.441152
speed         70.034568
dtype: float64

In [38]:
# define output
team_output = ['pokemon_id', 'poke_name', 'form', 'generation', 'type_1', 'type_2', 'hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed']

# creating df to randomly select a pokemon from the previously made dfs
team_df= [
    hp_poke.sample(n=1)[team_output],
    atk_poke.sample(n=1)[team_output],
    def_poke.sample(n=1)[team_output],
    sp_atk_poke.sample(n=1)[team_output],
    sp_def_poke.sample(n=1)[team_output],
    speed_poke.sample(n=1)[team_output]
]

#concat all information together
team = pd.concat(team_df, ignore_index=True)

print('6-Pokemon Team:')
team

6-Pokemon Team:


,pokemon_id,poke_name,form,generation,type_1,type_2,hp,attack,defense,sp_attack,sp_defense,speed
0,PKM1200,Munkidori,,9,Poison,Psychic,88,75,66,130,90,106
1,PKM1042,Dracozolt,,8,Electric,Dragon,90,100,90,80,70,75
2,PKM1002,Sandaconda,,8,Ground,,72,107,125,65,70,71
3,PKM0770,Clawitzer,,6,Water,,71,73,88,120,89,59
4,PKM0121,Starmie,,1,Water,Psychic,60,75,85,100,85,115
5,PKM0671,Keldeo,Ordinary Form,5,Water,Fighting,91,72,90,129,90,108


### Max Stats by Generation

In [29]:
# Highest HP Pokemon by Generation

max_hp_gen = df.groupby('generation')['hp'].transform('max')
max_hp = df[df['hp'] == max_hp_gen][['generation', 'pokemon_id', 'pokedex_number', 'poke_name', 'hp']]

max_hp

,generation,pokemon_id,pokedex_number,poke_name,hp
112,1,PKM0113,113,Chansey,250
241,2,PKM0242,242,Blissey,255
320,3,PKM0321,321,Wailord,170
435,4,PKM0436,426,Drifblim,150
501,4,PKM0502,487,Giratina,150
502,4,PKM0503,487,Giratina,150
612,5,PKM0613,594,Alomomola,165
798,6,PKM0799,716,Xerneas,126
799,6,PKM0800,717,Yveltal,126
914,7,PKM0915,799,Guzzlord,223


In [30]:
# Highest Attack Pokemon by Generation

max_attack_gen = df.groupby('generation')['attack'].transform('max')
max_attack = df[df['attack'] == max_attack_gen][['generation', 'pokemon_id', 'pokedex_number', 'poke_name', 'attack']]

max_attack

,generation,pokemon_id,pokedex_number,poke_name,attack
148,1,PKM0149,149,Dragonite,134
247,2,PKM0248,248,Tyranitar,134
389,3,PKM0390,386,Deoxys,180
414,4,PKM0415,409,Rampardos,165
669,5,PKM0670,646,Kyurem,170
688,6,PKM0689,150,Mewtwo,190
913,7,PKM0914,798,Kartana,181
1064,8,PKM1065,898,Calyrex,165
1144,9,PKM1145,964,Palafin,160


In [31]:
# Highest Defense Pokemon by Generation

max_defense_gen = df.groupby('generation')['defense'].transform('max')
max_defense = df[df['defense'] == max_defense_gen][['generation', 'pokemon_id', 'pokedex_number', 'poke_name', 'defense']]

max_defense

,generation,pokemon_id,pokedex_number,poke_name,defense
90,1,PKM0091,91,Cloyster,180
212,2,PKM0213,213,Shuckle,230
379,3,PKM0380,377,Regirock,200
416,4,PKM0417,411,Bastiodon,168
581,5,PKM0582,563,Cofagrigus,145
691,6,PKM0692,208,Steelix,230
702,6,PKM0703,306,Aggron,230
923,7,PKM0924,805,Stakataka,211
1054,8,PKM1055,890,Eternatus,250
1214,9,PKM1215,1025,Pecharunt,160


In [32]:
# Highest Sp Attack Pokemon by Generation

max_sp_attack_gen = df.groupby('generation')['sp_attack'].transform('max')
max_sp_attack = df[df['sp_attack'] == max_sp_attack_gen][['generation', 'pokemon_id', 'pokedex_number', 'poke_name', 'sp_attack']]

max_sp_attack

,generation,pokemon_id,pokedex_number,poke_name,sp_attack
149,1,PKM0150,150,Mewtwo,154
195,2,PKM0196,196,Espeon,130
389,3,PKM0390,386,Deoxys,180
497,4,PKM0498,483,Dialga,150
498,4,PKM0499,484,Palkia,150
668,5,PKM0669,646,Kyurem,170
689,6,PKM0690,150,Mewtwo,194
911,7,PKM0912,796,Xurkitree,173
1065,8,PKM1066,898,Calyrex,165
1177,9,PKM1178,994,Iron Moth,140


In [33]:
# Highest Sp Defense Pokemon by Generation

max_sp_defense_gen = df.groupby('generation')['sp_defense'].transform('max')
max_sp_defense = df[df['sp_defense'] == max_sp_defense_gen][['generation', 'pokemon_id', 'pokedex_number', 'poke_name', 'sp_defense']]

max_sp_defense

,generation,pokemon_id,pokedex_number,poke_name,sp_defense
143,1,PKM0144,144,Articuno,125
212,2,PKM0213,213,Shuckle,230
380,3,PKM0381,378,Regice,200
485,4,PKM0486,476,Probopass,150
633,5,PKM0634,615,Cryogonal,135
715,6,PKM0716,382,Kyogre,160
862,7,PKM0863,748,Toxapex,142
1054,8,PKM1055,890,Eternatus,250
1170,9,PKM1171,987,Flutter Mane,135
1185,9,PKM1186,1001,Wo-Chien,135


In [34]:
# Highest Speed Pokemon by Generation

max_speed_gen = df.groupby('generation')['speed'].transform('max')
max_speed = df[df['speed'] == max_speed_gen][['generation', 'pokemon_id', 'pokedex_number', 'poke_name', 'speed']]

max_speed

,generation,pokemon_id,pokedex_number,poke_name,speed
100,1,PKM0101,101,Electrode,150
168,2,PKM0169,169,Crobat,130
391,3,PKM0392,386,Deoxys,180
508,4,PKM0509,492,Shaymin,127
635,5,PKM0636,617,Accelgor,145
681,6,PKM0682,65,Alakazam,150
687,6,PKM0688,142,Aerodactyl,150
910,7,PKM0911,795,Pheromosa,151
1059,8,PKM1060,894,Regieleki,200
1174,9,PKM1175,991,Iron Bundle,136


In [36]:
# Creating df for pokemon with max stats for each generation

stats = ['hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed']

results = {}

for stat in stats:
    max_stats = df.groupby('generation')[stat].transform('max')
    max_pokemon = df[df[stat] == max_stats][['generation', 'pokemon_id', 'pokedex_number', 'poke_name', stat]]
    results[stat] = max_pokemon

max_df = pd.concat(results, names=['stat']).reset_index(level=0).reset_index(drop=True)

max_df

,stat,generation,pokemon_id,pokedex_number,poke_name,hp,attack,defense,sp_attack,sp_defense,speed
0,hp,1,PKM0113,113,Chansey,250.0,NaN,NaN,NaN,NaN,NaN
1,hp,2,PKM0242,242,Blissey,255.0,NaN,NaN,NaN,NaN,NaN
2,hp,3,PKM0321,321,Wailord,170.0,NaN,NaN,NaN,NaN,NaN
3,hp,4,PKM0436,426,Drifblim,150.0,NaN,NaN,NaN,NaN,NaN
4,hp,4,PKM0502,487,Giratina,150.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
56,speed,6,PKM0682,65,Alakazam,NaN,NaN,NaN,NaN,NaN,150.0
57,speed,6,PKM0688,142,Aerodactyl,NaN,NaN,NaN,NaN,NaN,150.0
58,speed,7,PKM0911,795,Pheromosa,NaN,NaN,NaN,NaN,NaN,151.0
59,speed,8,PKM1060,894,Regieleki,NaN,NaN,NaN,NaN,NaN,200.0


### Randomizer to Select 6 Pokemon with the Highest Stats

In [37]:
df_stat = {
    'max_hp': max_hp,
    'max_attack': max_attack,
    'max_defense': max_defense,
    'max_sp_attack': max_sp_attack,
    'max_sp_defense': max_sp_defense,
    'max_speed': max_speed
}

random_pick = {stat: df.sample(n=1) for stat, df in df_stat.items()}

random_6_pick = pd.concat(random_pick.values(), ignore_index=True)

random_6_pick

,generation,pokemon_id,pokedex_number,poke_name,hp,attack,defense,sp_attack,sp_defense,speed
0,6,PKM0799,716,Xerneas,126.0,NaN,NaN,NaN,NaN,NaN
1,3,PKM0390,386,Deoxys,NaN,180.0,NaN,NaN,NaN,NaN
2,6,PKM0703,306,Aggron,NaN,NaN,230.0,NaN,NaN,NaN
3,3,PKM0390,386,Deoxys,NaN,NaN,NaN,180.0,NaN,NaN
4,4,PKM0486,476,Probopass,NaN,NaN,NaN,NaN,150.0,NaN
5,6,PKM0688,142,Aerodactyl,NaN,NaN,NaN,NaN,NaN,150.0
